In [7]:
import tensorflow as tf
print("TF GPUs:", tf.config.list_physical_devices("GPU"))


TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


W0000 00:00:1773311143.403325  248064 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


# 00_env: 공통 환경 (거의 안 건드림)

In [1]:
import time, os, joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.datasets import reuters

I0000 00:00:1773310409.511039  248064 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# 01_data: Reuters 로드 + 디코딩 (한 번만 실행)

In [2]:
num_words_for_load = 10000
(x_train_idx, y_train), (x_test_idx, y_test) = reuters.load_data(
    num_words=num_words_for_load, test_split=0.2
)

word_index = reuters.get_word_index(path="reuters_word_index.json")
index_to_word = {index + 3: word for word, index in word_index.items()}
for index, token in enumerate(("<pad>", "<sos>", "<unk>")):
    index_to_word[index] = token

def decode_seq(seqs, index_to_word):
    return [" ".join(index_to_word.get(i, "<unk>") for i in s) for s in seqs]

x_train_text = decode_seq(x_train_idx, index_to_word)
x_test_text = decode_seq(x_test_idx, index_to_word)

print("Train:", len(x_train_text), "Test:", len(x_test_text))

Train: 8982 Test: 2246


# 02_helpers: 모델 팩토리 + Dense 생성 함수

In [3]:
def build_dense_model(
    input_dim,
    num_classes,
    hidden_units=[512, 128],   # 히든 레이어 크기 리스트
    dropout_rate=0.3,          # 0이면 Dropout 없음
    activation="relu",         # 히든 활성함수
):
    inputs = Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


In [4]:
def get_base_models(N_JOBS=4):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            solver="lbfgs",
            n_jobs=N_JOBS,
        ),
        "LinearSVC": LinearSVC(),
        "MultinomialNB": MultinomialNB(),
        "ComplementNB": ComplementNB(),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=None,
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=N_JOBS,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=N_JOBS,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            random_state=42,
            n_jobs=N_JOBS,
        ),
    }

def make_voting(N_JOBS=4):
    log_reg = LogisticRegression(
        max_iter=1000,
        multi_class="multinomial",
        solver="lbfgs",
        n_jobs=N_JOBS,
    )
    linear_svc = LinearSVC()
    mnb = MultinomialNB()

    voting_soft = VotingClassifier(
        estimators=[
            ("log_reg", log_reg),
            ("linear_svc", linear_svc),
            ("mnb", mnb),
        ],
        voting="soft",
    )
    return {"Voting": voting_soft}

def build_dense_model(input_dim, num_classes, hidden1=512, hidden2=128, dropout=0.3):
    inputs = Input(shape=(input_dim,))
    x = Dense(hidden1, activation="relu")(inputs)
    if dropout > 0:
        x = Dropout(dropout)(x)
    x = Dense(hidden2, activation="relu")(x)
    if dropout > 0:
        x = Dropout(dropout)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

# 03_config: 여기 값만 손으로 바꿔가며 실험

In [ ]:
vocab = 2000          # 500, 1000, 2000, 5000, None 등
N_JOBS = 6            # 2, 4, 6 정도
USE_DENSE = True

# --- Dense 하이퍼파라미터 ---
DENSE_HIDDEN = [512, 128]   # 예: [256, 64], [512, 256, 64] 등으로 바꿔보기
DENSE_DROPOUT = 0.3         # 0.0, 0.2, 0.5 등
DENSE_ACTIVATION = "relu"   # "relu", "tanh" 등
DENSE_EPOCHS = 5            # 3, 5, 10, 20 등
DENSE_BATCH_SIZE = 32       # 32, 64, 128 등

# --- 이번에 돌릴 전통 ML 모델 목록 ---
model_names_to_run = [
    "LogisticRegression",
    "LinearSVC",
    "MultinomialNB",
    "ComplementNB",
    "GradientBoosting",
    "Voting",
    # "RandomForest",
    # "XGBoost",
    # "LightGBM",
]

print("vocab:", vocab, "/ N_JOBS:", N_JOBS)
print("Dense:", DENSE_HIDDEN, "drop", DENSE_DROPOUT,
      "act", DENSE_ACTIVATION, "ep", DENSE_EPOCHS,
      "bs", DENSE_BATCH_SIZE)
print("models:", model_names_to_run)

vocab: 2000 / N_JOBS: 4
Dense: [512, 128] drop 0.3 act relu ep 5 bs 32
models: ['LogisticRegression', 'LinearSVC', 'MultinomialNB', 'ComplementNB', 'GradientBoosting', 'Voting']


## 04_run_experiment: 한 번 돌리기

In [6]:
results = []  # 연습장에서는 run할 때마다 새로 만들기

# 1) TF-IDF 만들기
vocab_label = "All" if vocab is None else str(vocab)

if vocab is None:
    dtmvector = CountVectorizer()
else:
    dtmvector = CountVectorizer(max_features=vocab)

tfidf_transformer = TfidfTransformer()

t0 = time.time()
x_train_dtm = dtmvector.fit_transform(x_train_text)
x_train_tfidf = tfidf_transformer.fit_transform(x_train_dtm)
tfidf_train_time = time.time() - t0

t0 = time.time()
x_test_dtm = dtmvector.transform(x_test_text)
x_test_tfidf = tfidf_transformer.transform(x_test_dtm)
tfidf_test_time = time.time() - t0

n_features = x_train_tfidf.shape[1]
print("TF-IDF shape:", x_train_tfidf.shape)

# 2) 전통 ML / 앙상블
base_models = get_base_models(N_JOBS=N_JOBS)
extra_models = make_voting(N_JOBS=N_JOBS)
all_models = {**base_models, **extra_models}

for model_name in model_names_to_run:
    model = all_models[model_name]
    print(f"\n[ML] Vocab={vocab_label}, Model={model_name}")

    t0 = time.time()
    model.fit(x_train_tfidf, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = model.predict(x_test_tfidf)
    test_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    model_size_kb = 0.0  # 연습장에서는 굳이 파일로 안 저장해도 됨

    results.append({
        "Vocabulary": vocab_label,
        "Model": model_name,
        "Accuracy": acc,
        "F1_macro": f1_macro,
        "TrainTime_s": train_time,
        "TestTime_s": test_time,
        "ModelSize_KB": model_size_kb,
        "NumFeatures": n_features,
        "TFIDF_TrainTime_s": tfidf_train_time,
        "TFIDF_TestTime_s": tfidf_test_time,
    })

# 3) DenseNN (옵션)
if USE_DENSE:
    print(f"\n[DL] Vocab={vocab_label}, Model=DenseNN")

    dense_model = build_dense_model(
        input_dim=n_features,
        num_classes=np.max(y_train) + 1,
        hidden_units=DENSE_HIDDEN,
        dropout_rate=DENSE_DROPOUT,
        activation=DENSE_ACTIVATION,
    )

    x_train_dense = x_train_tfidf.toarray()
    x_test_dense = x_test_tfidf.toarray()

    t0 = time.time()
    history = dense_model.fit(
        x_train_dense,
        y_train,
        epochs=DENSE_EPOCHS,
        batch_size=DENSE_BATCH_SIZE,
        validation_split=0.2,
        verbose=1,
    )
    train_time = time.time() - t0

    t0 = time.time()
    y_proba = dense_model.predict(x_test_dense, verbose=0)
    test_time = time.time() - t0

    y_pred = np.argmax(y_proba, axis=1)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    results.append({
        "Vocabulary": vocab_label,
        "Model": f"DenseNN_{DENSE_HIDDEN}_{DENSE_EPOCHS}ep",
        "Accuracy": acc,
        "F1_macro": f1_macro,
        "TrainTime_s": train_time,
        "TestTime_s": test_time,
        "ModelSize_KB": 0.0,
        "NumFeatures": n_features,
        "TFIDF_TrainTime_s": tfidf_train_time,
        "TFIDF_TestTime_s": tfidf_test_time,
        "NumParams": dense_model.count_params(),
    })

TF-IDF shape: (8982, 2000)

[ML] Vocab=2000, Model=LogisticRegression


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



[ML] Vocab=2000, Model=LinearSVC

[ML] Vocab=2000, Model=MultinomialNB

[ML] Vocab=2000, Model=ComplementNB

[ML] Vocab=2000, Model=GradientBoosting


KeyboardInterrupt: 

# 05_show_results: 이번 실험 결과 바로 확인


In [ ]:
results_df = pd.DataFrame(results)

print("\n=== 이번 실험 성능 요약 (Accuracy / F1_macro) ===")
display(
    results_df[["Model", "Accuracy", "F1_macro", "TrainTime_s"]]
    .sort_values("F1_macro", ascending=False)
)

print("\n=== 전체 raw 결과 ===")
display(results_df)